# Quantization Aware Training + Knowledge Distillation Benchmarking

In [7]:
import torch
import torch.onnx
import onnx
from onnx_tf.backend import prepare
from torch.ao.quantization.quantize_fx import convert_fx

from src.utils import load_data
from src.Quantization.utils.model_setup import setup_qat_student_model, quantization_mode
from src.utils import benchmark
from src.utils.model_setup import setup_model
from src.utils import test_inference, test_inference_onnx

2025-03-26 20:17:24.837823: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-26 20:17:24.844067: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743034644.849857   19680 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743034644.851591   19680 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743034644.857260   19680 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

ModuleNotFoundError: No module named 'tensorflow_addons'

### Load Original and Quantized model

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pretrained_weights = f"models/SkinCancer/Quantized/quantized_student_state.pth"
batch_size = 32
dataloaders = load_data(dataset="SkinCancer", batch_size=batch_size)
num_classes = len(dataloaders["train"].dataset.classes)
model = setup_qat_student_model(model_name="mobilenet_v2",num_classes=num_classes)

example_inputs = next(iter(dataloaders["train"]))[0].to(device)
student_model = quantization_mode(model, "fx", example_inputs=example_inputs)

# Move the model to CPU if needed (conversion is typically done on CPU).
student_model = student_model.to("cpu")

quantized_model = convert_fx(student_model)

# Now load the state dict.
state_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")
quantized_model.load_state_dict(state_dict)

# Set to eval mode.
quantized_model.eval()

teacher_model = setup_model("mobilenet_v2", None, num_classes)

Model prepared using FX Graph Mode QAT.


/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/ao/quantization/utils.py:407: UserWarning: must run observer before calling calculate_qparams. Returning default values.
  warnings.warn(
/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/_utils.py:392: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  device=storage.device,


### Perform Benchmarking (model_size, inference time, throughput, memory usage)

In [ ]:
device = torch.device("cpu")

benchmark(model1=teacher_model, model2=quantized_model, dataloader=dataloaders["test"], device=device)

In [5]:
device = torch.device("cpu")
test_inference(quantized_model, dataloaders["test"], device, None)

Inference Progress: 100%|██████████| 115/115 [00:19<00:00,  5.95it/s]

Metrics Results:
Accuracy: 0.7559
Recall: 0.6081
Precision: 0.7083
F1-Score: 0.6330
AUC-Score: 0.9667



/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [4]:
def export_model_to_onnx(model: torch.nn.Module, example_input: torch.Tensor, onnx_file_path: str, opset_version: int = 12) -> None:
    """
    Exports a given PyTorch model to the ONNX format.

    This function uses torch.onnx.export to convert the model to ONNX. It assumes the model is in eval mode.
    Note that quantized models may face compatibility issues with ONNX; ensure that the opset_version and model
    configuration are supported.

    Args:
        model (torch.nn.Module): The PyTorch model to export.
        example_input (torch.Tensor): An example input tensor with the appropriate shape.
        onnx_file_path (str): Path where the ONNX file will be saved.
        opset_version (int): ONNX opset version to use. Defaults to 12.
    """
    # Ensure the model is in evaluation mode.
    model.eval()

    # Export the model.
    torch.onnx.export(
        model,                          # model being exported
        example_input,                  # example input to the model
        onnx_file_path,                 # where to save the ONNX model
        export_params=True,             # store the trained parameter weights inside the model file
        opset_version=opset_version,    # specify the ONNX version to export the model to
        do_constant_folding=True,       # execute constant folding for optimization
        input_names=['input'],          # the model's input names
        output_names=['output'],        # the model's output names
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}  # enable dynamic batch size
    )
    print(f"Model successfully exported to {onnx_file_path}")

device = torch.device("cpu")
# Example usage:
# Given your code, you have quantized_model and example_inputs already defined.
onnx_export_path: str = "quantized_student.onnx"
export_model_to_onnx(quantized_model, example_inputs.to(device), onnx_export_path)

Model successfully exported to quantized_student.onnx


In [3]:

test_inference_onnx("quantized_student.onnx", dataloaders["test"], torch.device("cpu"), save_dir=None)


ONNX Inference Progress: 100%|██████████| 115/115 [00:19<00:00,  5.76it/s]
/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


ONNX Metrics Results:
Accuracy: 0.7594
Recall: 0.6122
Precision: 0.6967
F1-Score: 0.6335
AUC-Score: 0.9667


In [ ]:
# Load the ONNX model.
onnx_model = onnx.load("quantized_student.onnx")
# Convert to TensorFlow representation.
tf_rep = prepare(onnx_model)
# Export the model as a SavedModel.
tf_rep.export_graph("model_tf")
